In [1]:
import random
from rdkit import Chem
from molpher.core import MolpherMol, MolpherAtom
from molpher.core.morphing.operators import MorphingOperator
from rdkit.Chem.EnumerateStereoisomers import EnumerateStereoisomers, StereoEnumerationOptions
from rdkit.Chem import rdChemReactions
from rdkit.Chem import rdmolops
from rdkit.Chem import Descriptors  
from molpher.core import ExplorationTree as ETree

class OxidizeAldehydeToAcid(MorphingOperator):
    def __init__(self):
        super(OxidizeAldehydeToAcid, self).__init__()
        self._name = "Oxidize Aldehyde to Acid"
        self._target_carbons = [] 
        self.PATTERN = Chem.MolFromSmarts("[CX3H1](=O)[#6,#1]")

    def setOriginal(self, mol):
        super(OxidizeAldehydeToAcid, self).setOriginal(mol)
        self._target_carbons = []
        
        if not self.original:
            return
            
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: 
            return
            
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            self._target_carbons.append(match[0])

    def morph(self):
        if not self.original: 
            return None
            
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: 
            return None
        
        if not self._target_carbons:
            return MolpherMol(other=rdkit_mol)
            
        idx_c = random.choice(self._target_carbons)
        
        try:
            rw_mol = Chem.RWMol(rdkit_mol)
            
            new_o_idx = rw_mol.AddAtom(Chem.Atom(8))
            
            rw_mol.AddBond(idx_c, new_o_idx, Chem.BondType.SINGLE)
            
            new_mol = rw_mol.GetMol()
            
            for idx in [idx_c, new_o_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
        
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
            
        except Exception as e:
            return MolpherMol(other=rdkit_mol)
        
    def getName(self):
        return self._name

class AminoAcidConjugation(MorphingOperator):
    def __init__(self):
        super(AminoAcidConjugation, self).__init__()
        self._name = "Amino Acid Conjugation (Gly/Tau/Gln)"
        self._target_atoms = []
        self.PATTERN = Chem.MolFromSmarts("[CX3](=O)[OX2H]")

        self.TEMPLATES = {
            "Glycine": Chem.MolFromSmiles("NCC(=O)O"),
            "Taurine": Chem.MolFromSmiles("NCCS(=O)(=O)O"),
            "Glutamine": Chem.MolFromSmiles("N[C@@H](CCC(=O)N)C(=O)O") # Διατήρηση στερεοχημείας
        }

    def setOriginal(self, mol):
        super(AminoAcidConjugation, self).setOriginal(mol)
        self._target_atoms = []
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return
        
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            self._target_atoms.append((match[0], match[2]))

    def morph(self):
        if not self.original or not self._target_atoms:
            return MolpherMol(other=self.original.asRDMol())
        
        rdkit_mol = self.original.asRDMol()
        try:
            chosen_c_idx, chosen_oh_idx = random.choice(self._target_atoms)
            template_name = random.choice(list(self.TEMPLATES.keys()))
            template_mol = self.TEMPLATES[template_name]
            
            combined = Chem.CombineMols(rdkit_mol, template_mol)
            rw_combined = Chem.RWMol(combined)
            
            n_amino_idx = rdkit_mol.GetNumAtoms()
            
            rw_combined.AddBond(chosen_c_idx, n_amino_idx, Chem.BondType.SINGLE)
            
            rw_combined.RemoveAtom(chosen_oh_idx)
            
            new_mol = rw_combined.GetMol()

            for atom in new_mol.GetAtoms():
                if atom.GetAtomicNum() in [6, 7]: # Άνθρακας καρβονυλίου και Άζωτο αμιδίου
                    atom.SetNoImplicit(False)
                    atom.SetNumExplicitHs(0)
                    atom.SetFormalCharge(0)
            
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
            
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name


oxidize_aldehyde_op = OxidizeAldehydeToAcid()
amino_op = AminoAcidConjugation()

class FindClosest:
    def __init__(self):
        self.closest_mol = None
        self.closest_distance = None
        
    def __call__(self, morph):
        if not self.closest_mol or self.closest_distance > morph.dist_to_target:
            self.closest_mol = morph
            self.closest_distance = morph.dist_to_target
closest_info = FindClosest()

closest_info = FindClosest()

start_mol = MolpherMol("O=Cc1ccccc1")       # Βενζαλδεΰδη
target_mol = MolpherMol("O=C(NCC(=O)O)c1ccccc1")  # Ιππουρικό οξύ
tree = ETree.create(source=start_mol, target=target_mol)
tree.morphing_operators = (oxidize_aldehyde_op, amino_op)

print("--- STARTING MOLPHER SEARCH TREE ---")
max_generations = 40
while not tree.path_found and tree.generation_count < max_generations:
    tree.generateMorphs()
    tree.sortMorphs()
    tree.filterMorphs()
    tree.extend()
    tree.prune()
    tree.traverse(closest_info)
    
    print(f"Generation #{tree.generation_count}")
    print(f"Molecules in tree: {tree.mol_count}")
    if closest_info.closest_mol:
        print(f"Closest to target: {closest_info.closest_mol.getSMILES()} (Distance: {closest_info.closest_distance:.4f})")
    print("-" * 40)

--- STARTING MOLPHER SEARCH TREE ---
Generation #1
Molecules in tree: 2
Closest to target: O=C(O)C1=CC=CC=C1 (Distance: 0.4828)
----------------------------------------
Generation #2
Molecules in tree: 5
Closest to target: O=C(O)CNC(=O)C1=CC=CC=C1 (Distance: 0.0000)
----------------------------------------
